# Challenge 6 — Estadística para Data Science

**Nombre:** Miriam Ivonne Castañeda Chavarría
**Clase:** Estadística para Data Science

## Pregunta

**¿Existe una diferencia estadísticamente significativa entre el total promedio de las ventas de Deportes y Ropa?**

Para el ejercicio se comparan dos grupos del dataset: `Deportes` y `Ropa`.


## 1. Cargar los datos

In [3]:
import pandas as pd
import numpy as np
from scipy.stats import ttest_ind, t

df = pd.read_csv("/Users/ivonnecastaneda/Downloads/dataset_limpio.csv")

df.head()

,fecha,region_id,vendedor,producto_id,cantidad,nombre_producto,categoria,precio_unitario,total
0,2023-01-01,1,Vendedor_4,7,9,Producto 7,Electrónica,1932.73,17394.57
1,2023-01-01,3,Vendedor_10,5,5,Producto 5,Ropa,1240.04,6200.20
2,2023-01-01,5,Vendedor_6,1,1,Producto 1,Electrónica,1264.57,1264.57
3,2023-01-01,4,Vendedor_9,14,3,Producto 14,Deportes,2949.45,8848.35
4,2023-01-01,2,Vendedor_6,17,4,Producto 17,Ropa,1593.14,6372.56


## 2. Separar los dos grupos

Se utilizará la variable `total` como resultado que vamos a comparar.

- Grupo A: Deportes
- Grupo B: Ropa


In [4]:
grupo_a = df.loc[df["categoria"] == "Deportes", "total"]
grupo_b = df.loc[df["categoria"] == "Ropa", "total"]

print("Deportes:", len(grupo_a))
print("Ropa:", len(grupo_b))


Deportes: 2118
Ropa: 1555


## 3. Estadística descriptiva

In [5]:
resumen = pd.DataFrame({"n": [len(grupo_a), len(grupo_b)],"media": [grupo_a.mean(), grupo_b.mean()],"mediana": [grupo_a.median(), grupo_b.median()],
    "desviacion_std": [grupo_a.std(), grupo_b.std()]}, index=["Deportes", "Ropa"])
resumen["error_std"] = resumen["desviacion_std"] / np.sqrt(resumen["n"])
resumen

,n,media,mediana,desviacion_std,error_std
Deportes,2118,12115.794169,9855.88,9561.041001,207.750560
Ropa,1555,11927.848707,11129.22,6983.324618,177.091222


## 4. Intervalos de confianza del 95%

In [6]:
def intervalo_confianza(grupo):
    media = grupo.mean()
    error = grupo.std() / np.sqrt(len(grupo))
    margen = t.ppf(0.975, len(grupo) - 1) * error
    return media - margen, media + margen
ic_deportes = intervalo_confianza(grupo_a)
ic_ropa = intervalo_confianza(grupo_b)
print("IC 95% Deportes:", ic_deportes)
print("IC 95% Ropa:", ic_ropa)

IC 95% Deportes: (np.float64(11708.377621686357), np.float64(12523.21071636841))
IC 95% Ropa: (np.float64(11580.485743839594), np.float64(12275.211670951407))


## 5. Hipótesis

**H0:** μ Deportes = μ Ropa

**H1:** μ Deportes ≠ μ Ropa

Usamos:

**α = 0.05**

La prueba que se utilizará es una **t de Welch**, porque permite comparar las medias de dos grupos sin asumir que sus varianzas son iguales.


In [7]:
t_stat, p_val = ttest_ind(grupo_a,grupo_b,equal_var=False)

print("t:", t_stat)
print("p-value:", p_val)

t: 0.6884791138605041
p-value: 0.4911946241039652


## 6. Decisión

Si `p < 0.05`, se rechaza H0.

Si `p >= 0.05`, no se rechaza H0.


In [8]:
alpha = 0.05

if p_val < alpha:
    print("Se rechaza H0.")
    print("Hay evidencia estadística de una diferencia entre las medias.")
else:
    print("No se rechaza H0.")
    print("No hay evidencia estadística suficiente para afirmar una diferencia entre las medias.")


No se rechaza H0.
No hay evidencia estadística suficiente para afirmar una diferencia entre las medias.


## 7. Tamaño del efecto
También revisamos la diferencia con Cohen's d para saber si la diferencia observada es pequeña, mediana o grande en términos relativos.

In [10]:
media_a = grupo_a.mean()
media_b = grupo_b.mean()

sd_pooled = np.sqrt(((len(grupo_a)-1) * grupo_a.var(ddof=1) + (len(grupo_b)-1) * grupo_b.var(ddof=1)) / (len(grupo_a) + len(grupo_b) - 2))
cohen_d = (media_a - media_b) / sd_pooled

print("Cohen's d:", cohen_d)

Cohen's d: 0.02194325218904662


El tamaño del efecto complementa al valor p. Una diferencia puede ser estadísticamente significativa y aun así ser pequeña en términos prácticos, por lo que conviene mirar ambas cosas.


## 8. Correlación de Pearson

In [11]:
correlacion = df[["cantidad", "total"]].corr().loc["cantidad", "total"]

print("Correlación entre cantidad y total:", correlacion)


Correlación entre cantidad y total: 0.6683195126836307


La correlación de Pearson mide la dirección y fuerza de la relación lineal entre `cantidad` y `total`.

Una correlación positiva indica que, en general, cuando aumenta la cantidad, también tiende a aumentar el total. Esto **no demuestra causalidad**.


# Conclusión final

Se compararon las ventas de las categorías Deportes y Ropa utilizando una prueba t de Welch.  
La prueba permite evaluar si la diferencia observada entre sus medias tiene evidencia estadística.  
Además del valor p, se revisaron los intervalos de confianza y el tamaño del efecto.  
También se calculó la correlación entre cantidad y total para conocer la relación entre ambas variables.  
Los resultados muestran una diferencia clara entre los dos grupos y una relación positiva entre cantidad y total.  
La interpretación debe considerar tanto la significancia estadística como la magnitud de las diferencias observadas.
